# Fine-tuning the detector — the unblock

Everything else in this project is downstream of one problem: **the pipeline has only ever run on COCO weights, which have no `ball`, `goalkeeper` or `referee` class.**

That is not a quality issue, it is dead code. In `src/pipeline.py`:

- `ball_ids` resolves to `[]`, so `ball_xy` is always `None`
- possession is only computed inside `if ball_xy is not None`, so it never increments
- `stats["possession_pct"]` is therefore always `None`

**Two of the three headline features have never produced a number.**

There is a second, subtler win. Team assignment uses k-means with k=2, which *cannot* represent goalkeepers and referees — they wear a third and fourth kit and get forced into an outfield team. Detecting them as their own classes removes them from the clustering input entirely. That is a structural fix, not a tuning one: no value of any hyperparameter makes k=2 correct, but changing what enters the clustering does.

**Runtime:** ~30–50 min on a T4. Set **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# Pins carry the same hard-won reasons as the rest of the project:
#   numpy<2.4       -- 2.4 removed np.cross for 2-D vectors, which supervision uses
#   supervision<0.30 -- ByteTrack was removed in 0.30
!pip install -q "ultralytics>=8.4,<9" "supervision>=0.29,<0.30" "numpy<2.4" roboflow

import torch, ultralytics
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"

## 1. The dataset

[Roboflow's football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) labels four classes: `ball`, `goalkeeper`, `player`, `referee`.

You need a free Roboflow API key (Settings → API keys). Paste it into Colab's secrets as `ROBOFLOW_API_KEY`, or into the prompt below.

**Check the version number on the Universe page before running** — it increments as the dataset is updated, and an outdated number fails with an unhelpful error.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass.getpass("Roboflow API key: ")

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)

WORKSPACE = "roboflow-jvuqo"
PROJECT   = "football-players-detection-3zvbc"
VERSION   = 12          # <-- verify this on the Universe page

dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
DATA_YAML = os.path.join(dataset.location, "data.yaml")
print("\ndata.yaml ->", DATA_YAML)

## 2. Verify the class ordering before training anything

This is the check that would have saved the previous project an entire run.

A class *index* is only meaningful relative to the model that produced it. In COCO, index 0 is `person`. In this dataset, index 0 is almost certainly `ball` — the classes are alphabetical. Any code that hardcodes `class_id == 0` to mean "player" would silently track the ball as a squad of players and still emit a full set of plausible statistics.

`src/pipeline.py` already resolves classes **by name** via `class_ids_for()`, so it is immune. Print the mapping anyway — confirming the thing you believe is cheap, and this is the exact assumption that invalidated the Traffic Lens baseline.

In [ ]:
import yaml
cfg = yaml.safe_load(open(DATA_YAML))
names = cfg["names"]
names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else names

print("class index -> name")
for i, n in sorted(names.items()):
    print(f"  {i}: {n}")

# The same name-sets pipeline.py uses. If these resolve to nothing, the pipeline
# would fail loudly later -- better to find out now, before an hour of training.
PLAYER_NAMES = {"player", "person", "goalkeeper", "goalkeepers"}
BALL_NAMES   = {"ball", "sports ball", "football", "soccer ball"}
REFEREE_NAMES = {"referee", "refere"}

for label, wanted in [("players", PLAYER_NAMES), ("ball", BALL_NAMES), ("referee", REFEREE_NAMES)]:
    got = [i for i, n in names.items() if str(n).lower() in wanted]
    print(f"{label:9s} -> indices {got}  ({[names[i] for i in got]})")
    assert got, f"no class matched {label}! pipeline.py would not find it either."
print("\nAll three resolve. The pipeline will find them by name.")

## 3. Train

`imgsz=1280` rather than the usual 640, for the reason the whole project keeps running into: **a football is ~10 px in a 1080p frame and effectively disappears at 640.** Training at the resolution you will infer at matters more here than in most fine-tunes.

That costs roughly 4× the compute of 640. It is the single biggest knob on runtime — drop to 640 if you only care about players, but expect the ball class to be near-useless.

50 epochs because the previous project's 20-epoch run stopped with the loss curve still falling — training ended because the cosine schedule had decayed the learning rate to zero, not because the model had converged. That was a budget artifact reported as a result. Watch the curve, not the epoch count.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=1280,       # the ball is why
    batch=8,          # 1280 is memory-hungry; drop to 4 if you OOM on a T4
    patience=15,      # stop if val stops improving, rather than burning the budget
    project="runs",
    name="soccer_yolo26n_1280",
    plots=True,
)
print("\nweights ->", results.save_dir)

## 4. Per-class metrics — never read the mean alone

The `m` in mAP is "mean over classes", and it is where the most common misreading lives. **Ball detection will be dramatically worse than player detection** — roughly 10 px versus 200 px — and a single averaged number hides exactly the failure you most need to see.

Expect something like: player AP high, goalkeeper and referee moderate (few instances), ball much lower. If the headline mAP50 looks respectable, it is being carried by the `player` class, which is 90% of the boxes.

Report per class. Always.

In [ ]:
best = f"{results.save_dir}/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=DATA_YAML, imgsz=1280, split="val")

print(f"\n{'class':<14}{'AP50':>9}{'AP50-95':>10}")
print("-" * 33)
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{m.names[int(c)]:<14}{metrics.box.ap50[i]:>9.3f}{metrics.box.ap[i]:>10.3f}")
print("-" * 33)
print(f"{'MEAN':<14}{metrics.box.map50:>9.3f}{metrics.box.map:>10.3f}")
print("\nIf 'ball' is far below the mean, that is the expected result, not a bug.")
print("It is also the number that decides whether possession is trustworthy.")

## 5. Run the real pipeline with the new weights

The moment of truth. With a `ball` class present, `ball_xy` stops being `None` and **possession is computed for the first time in this project's history**.

Two things to look for in the stats below:

- `ball_class_available: true` — the fine-tune worked
- `possession_pct` is a dict rather than `None` — the feature is alive

Without `keypoints`, `has_pitch_mapping` stays false and you get tracking and team colours but no minimap, distances or possession. Supply four or more pitch correspondences to unlock those — and note that with five or more, the homography can finally report its own reprojection error in metres.

In [ ]:
!git clone -q https://github.com/akshay131996/soccer-analytics.git || (cd soccer-analytics && git pull -q)
import sys, json
sys.path.insert(0, "soccer-analytics/src")

from pipeline import process_video

SOURCE = ""   # <-- path to a soccer clip. Upload one, or point at your own footage.
assert SOURCE, "Set SOURCE to a video path first."

stats = process_video(
    source=SOURCE,
    output_path="annotated.mp4",
    weights=best,
    keypoints=None,        # supply 5+ correspondences to unlock metres + a residual
    imgsz=1280,
    max_frames=250,        # a short run first; remove once it works
)
print(json.dumps(stats, indent=2))

## 6. Keep the artifacts

The weights are the thing worth keeping — they are what makes every downstream feature real, and retraining costs another GPU hour. `best.pt` for a nano model is only a few MB, small enough to commit.

In [ ]:
import shutil, os
os.makedirs("artifacts", exist_ok=True)
shutil.copy(best, "artifacts/yolo26n_soccer_finetuned.pt")
for f in ["results.png", "results.csv", "BoxPR_curve.png", "confusion_matrix_normalized.png"]:
    p = os.path.join(results.save_dir, f)
    if os.path.exists(p):
        shutil.copy(p, f"artifacts/{f}")
if os.path.exists("annotated.mp4"):
    shutil.copy("annotated.mp4", "artifacts/annotated.mp4")

shutil.make_archive("soccer_artifacts", "zip", "artifacts")
print("\n".join(sorted(os.listdir("artifacts"))))
try:
    from google.colab import files
    files.download("soccer_artifacts.zip")
except Exception:
    print("\nNot on Colab — grab soccer_artifacts.zip manually.")